6  E-Commerce Recommendation Engine

In [5]:
import pandas as pd
import numpy as np

# Generate a synthetic dataset for demonstration
np.random.seed(42)

num_users = 100
num_items = 50
num_interactions = 1000

user_ids = np.random.randint(1, num_users + 1, num_interactions)
item_ids = np.random.randint(1, num_items + 1, num_interactions)
# Assuming 'rating' as a form of interaction, e.g., 1-5 stars or implicit feedback
ratings = np.random.randint(1, 6, num_interactions)

data = pd.DataFrame({
    'user_id': user_ids,
    'item_id': item_ids,
    'rating': ratings
})

print("Generated Interaction Data:")
display(data.head())
print(f"\nTotal interactions: {len(data)}")
print(f"Total unique users: {data['user_id'].nunique()}")
print(f"Total unique items: {data['item_id'].nunique()}")

Generated Interaction Data:


,user_id,item_id,rating
0,52,34,4
1,93,47,2
2,15,8,2
3,72,40,2
4,61,49,2



Total interactions: 1000
Total unique users: 100
Total unique items: 50


In [6]:
# Create a user-item interaction matrix
# We'll pivot the table to have users as index, items as columns, and ratings as values
user_item_matrix = data.pivot_table(index='user_id', columns='item_id', values='rating')

print("User-Item Interaction Matrix (first 5 rows and columns):")
display(user_item_matrix.iloc[:5, :5])

print(f"\nShape of user-item matrix: {user_item_matrix.shape}")
print("Note: NaN values represent items a user has not interacted with.")

User-Item Interaction Matrix (first 5 rows and columns):


item_id,1,2,3,4,5
user_id,,,,,
1,NaN,NaN,2.0,NaN,NaN
2,NaN,NaN,NaN,4.0,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,1.0,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN



Shape of user-item matrix: (100, 50)
Note: NaN values represent items a user has not interacted with.


In [7]:
from sklearn.model_selection import train_test_split

# To perform train-test split effectively for recommendation systems,
# we need to consider the interactions themselves, not just the matrix.
# We'll split the original 'data' DataFrame.

# First, let's ensure we only consider actual interactions (non-NaN ratings if applicable)
# Our synthetic data already only contains explicit ratings, so no NaN values here initially.

# Split the data into training and testing sets
# We use a stratify parameter if we want to maintain the proportion of certain categories,
# but for a general recommendation system with many users/items, a simple split is often used first.

train_df, test_df = train_test_split(data, test_size=0.2, random_state=42)

print("Shape of training data:", train_df.shape)
print("Shape of testing data:", test_df.shape)

# Now, we can reconstruct user-item matrices for training and testing
# The training matrix will be used to train the model
# The testing matrix will be used to evaluate predictions

train_user_item_matrix = train_df.pivot_table(index='user_id', columns='item_id', values='rating')
test_user_item_matrix = test_df.pivot_table(index='user_id', columns='item_id', values='rating')

print("\nTraining User-Item Matrix (first 5 rows and columns):")
display(train_user_item_matrix.iloc[:5, :5])

print("\nTesting User-Item Matrix (first 5 rows and columns):")
display(test_user_item_matrix.iloc[:5, :5])

print("\nNote: Both matrices will contain NaN values for unobserved interactions.")

Shape of training data: (800, 3)
Shape of testing data: (200, 3)

Training User-Item Matrix (first 5 rows and columns):


item_id,1,2,3,4,5
user_id,,,,,
1,NaN,NaN,2.0,NaN,NaN
2,NaN,NaN,NaN,4.0,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN



Testing User-Item Matrix (first 5 rows and columns):


item_id,1,2,4,5,6
user_id,,,,,
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,1.0,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,2.0



Note: Both matrices will contain NaN values for unobserved interactions.


In [8]:
# Install the surprise library if you haven't already
!pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 46.5 MB/s eta 0:00:00


In [9]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

# The Reader object is used to parse a file containing ratings.
# It specifies the rating scale (e.g., from 1 to 5).
reader = Reader(rating_scale=(1, 5))

# Load the dataset from the pandas DataFrame `data`
# The DataFrame must have three columns: user_id, item_id, and rating (in that order).
# `data` DataFrame already has these columns.

# We will use the full `data` DataFrame here to load into Surprise's Dataset
# as Surprise's train_test_split handles it internally.
# If we wanted to strictly use our `train_df` and `test_df` we created earlier,
# we would load them separately into Dataset objects.

# For simplicity and to leverage Surprise's built-in splitting and cross-validation,
# we'll load the full dataset and let Surprise handle the split.
# Alternatively, we could load from train_df and test_df if we want to ensure consistency
# with the `sklearn.model_selection.train_test_split` done previously.

# Let's use our previously created train_df to build the training set for Surprise.
# This ensures consistency with the previous data splitting step.
trainset_surprise = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader).build_full_trainset()

# The testset will be a list of tuples (user, item, true_rating)
# which can be directly used by Surprise's test method.
# We transform the test_df into this format.
testset_surprise = list(test_df.itertuples(index=False, name=None))

print("Surprise training set created with", trainset_surprise.n_users, "users and", trainset_surprise.n_items, "items.")
print("Surprise test set created with", len(testset_surprise), "ratings.")

# Train the SVD model
algo = SVD(random_state=42)
algo.fit(trainset_surprise)

print("\nSVD model training complete.")

Surprise training set created with 100 users and 50 items.
Surprise test set created with 200 ratings.

SVD model training complete.
